In [11]:
import os
#os.chdir("../")
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/vitaliy98765321/DS_edep.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="vitaliy98765321"
os.environ["MLFLOW_TRACKING_PASSWORD"]="2446e8acfebd73b82814a0460868667ea4e1a096"

In [12]:
%pwd

'c:\\Users\\QSUS\\Desktop\\6_sem\\course\\EDEP\\DS_edep'

In [13]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluatingConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    mlflow_uri: str
    all_params: dict
    metric_file_name: Path
    target_column: str

In [14]:
from src.ds_edep.constants import *
from src.ds_edep.utils.common import read_yaml, save_json, create_directories

In [15]:
class ConfigManager:
    def __init__(self,
                 config_path = CONFIG_FILE_PATH,
                 params_path = PARAMS_FILE_PATH,
                 schema_path = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_path)
        self.schema = read_yaml(schema_path)
        
        create_directories([self.config.artifacts_root])
    
    def get_model_evaluating_config(self) -> ModelEvaluatingConfig:
        config=self.config.model_evaluation
        params=self.params.model_param
        schema=self.schema
        
        create_directories([config.root_dir])
        
        obj_config = ModelEvaluatingConfig(
            root_dir = config.root_dir,
            test_data_path = config.test_data_path,
            model_path = config.model_path,
            mlflow_uri = config.mlflow_uri,
            all_params = params,
            metric_file_name = config.metric_file_name,
            target_column = schema.TARGET_COLUMN.name
        )
        
        return obj_config

In [16]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

In [17]:
class ModelEvaluating:
    def __init__(self, config: ModelEvaluatingConfig):
        self.config = config
        
    def evaluate_model(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2
    
    def log_metrics_mlflow(self):
        cfg = self.config
        test_data = pd.read_csv(cfg.test_data_path)
        model = joblib.load(cfg.model_path)
        
        X_test = test_data.drop(columns= [cfg.target_column])
        y_test = test_data[cfg.target_column]
        
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            predict_vals = model.predict(X_test)
            rmse, mae, r2 = self.evaluate_model(y_test, predict_vals)
            
            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)
            
            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("r2", r2)
            mlflow.log_metric("mae", mae)
            
            if tracking_url_type_store != "file":
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticModel")
            else:
                mlflow.sklearn.log_model(model, "model")

In [18]:
try:
    manager = ConfigManager()
    cfg = manager.get_model_evaluating_config()
    evaluator = ModelEvaluating(cfg)
    evaluator.log_metrics_mlflow()
    
except Exception as e:
    raise e

[2026-06-09 11:32:49,663: INFO: common: created directory at: artifacts]
[2026-06-09 11:32:49,665: INFO: common: created directory at: artifacts/model_evaluation]


[2026-06-09 11:32:51,138: INFO: common: JSON file saved succesfully to artifacts\model_evaluation\metrics.json]


2026/06/09 11:32:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 11:32:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'ElasticModel'.
2026/06/09 11:33:09 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticModel, version 1
Created version '1' of model 'ElasticModel'.


🏃 View run grandiose-kit-661 at: https://dagshub.com/vitaliy98765321/DS_edep.mlflow/#/experiments/0/runs/ed55b5981a71438da24a7a0140d87d0e
🧪 View experiment at: https://dagshub.com/vitaliy98765321/DS_edep.mlflow/#/experiments/0
